<a href="https://colab.research.google.com/github/ksrivithya/EGFR-QSAR-Portfolio/blob/main/Drug_Likeness_ML_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q rdkit scikit-learn pandas numpy

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors

print("Environment ready.")

Environment ready.


In [2]:
# Sample dataset: (compound_id, smiles, is_drug_like_label)
raw_data = [
    ("CMPD_01", "CC(=O)Oc1ccccc1C(=O)O", 1),          # Aspirin-like small molecule
    ("CMPD_02", "CN1CCC[C@H]1c2cccnc2", 1),         # Nicotine-like scaffold
    ("CMPD_03", "CC(C)Cc1ccc(C(C)C(=O)O)cc1", 1),   # Ibuprofen-like scaffold
    ("CMPD_04", "CCCCCCCCCCCCCCCC(=O)OCC(O)CO", 0),# Long fatty chain (higher MW/LogP)
    ("CMPD_05", "C1=CC=CC=C1-C1=CC=CC=C1.Cl", 1),    # Salt mixture (dot-separated)
    ("CMPD_06", "C1CC(=O)NC1", 1),                  # Valid small ring
    ("CMPD_07", "INVALID_SMILES_STRING", 0),        # Malformed SMILES -> returns None
]

df = pd.DataFrame(raw_data, columns=["compound_id", "raw_smiles", "drug_like"])
print("Raw rows:", len(df))

Raw rows: 7


In [3]:
def clean_molecule(smiles_str):
    if not isinstance(smiles_str, str) or not smiles_str.strip():
        return None, None

    # Keep largest fragment if salts/multicomponent SMILES are present
    if "." in smiles_str:
        fragments = smiles_str.split(".")
        smiles_str = max(fragments, key=len)

    mol = Chem.MolFromSmiles(smiles_str.strip())
    if mol is None:
        return None, None  # Edge case: invalid SMILES or valence error

    can_smiles = Chem.MolToSmiles(mol, canonical=True)
    return mol, can_smiles

cleaned_rows = []
for _, row in df.iterrows():
    mol, can_smiles = clean_molecule(row["raw_smiles"])
    if mol is not None:
        cleaned_rows.append({
            "compound_id": row["compound_id"],
            "canonical_smiles": can_smiles,
            "drug_like": row["drug_like"],
            "mol": mol
        })

clean_df = pd.DataFrame(cleaned_rows).drop_duplicates(subset=["canonical_smiles"]).reset_index(drop=True)
print("Valid unique molecules:", len(clean_df))

Valid unique molecules: 6


In [4]:
descriptor_records = []
for _, row in clean_df.iterrows():
    mol = row["mol"]
    descriptor_records.append({
        "mol_wt": Descriptors.MolWt(mol),
        "mol_logp": Descriptors.MolLogP(mol),
        "hbd": Lipinski.NumHDonors(mol),
        "hba": Lipinski.NumHAcceptors(mol),
        "tpsa": rdMolDescriptors.CalcTPSA(mol),
    })

desc_df = pd.DataFrame(descriptor_records)
feat_df = pd.concat([clean_df[["compound_id", "canonical_smiles", "drug_like"]], desc_df], axis=1)
print(feat_df[["compound_id", "mol_wt", "mol_logp", "hbd", "hba", "tpsa", "drug_like"]])

  compound_id   mol_wt  mol_logp  hbd  hba   tpsa  drug_like
0     CMPD_01  180.159    1.3101    1    3  63.60          1
1     CMPD_02  162.236    1.8483    0    2  16.13          1
2     CMPD_03  206.285    3.0732    1    1  37.30          1
3     CMPD_04  330.509    4.3641    2    4  66.76          0
4     CMPD_05  154.212    3.3536    0    0   0.00          1
5     CMPD_06   85.106   -0.1036    1    1  29.10          1


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

feature_cols = ["mol_wt", "mol_logp", "hbd", "hba", "tpsa"]
X = feat_df[feature_cols]
y = feat_df["drug_like"]

# Note: Use a larger dataset (e.g., 200+ rows) for meaningful train/test evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds, zero_division=0))

Test Accuracy: 1.0
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         2

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



In [6]:
import requests

# Fetch 200 approved or bioactive small molecules from ChEMBL REST API
url = "https://www.ebi.ac.uk/chembl/api/data/molecule.json?limit=200&molecule_type=Small%20molecule"
response = requests.get(url).json()

records = []
for m in response.get("molecules", []):
    structs = m.get("molecule_structures") or {}
    smiles = structs.get("canonical_smiles")
    if smiles:
        records.append({
            "chembl_id": m.get("molecule_chembl_id"),
            "raw_smiles": smiles,
            "max_phase": m.get("max_phase", 0)  # Clinical/approved phase (0 to 4)
        })

raw_df = pd.DataFrame(records)
print("Fetched molecules from ChEMBL:", len(raw_df))
raw_df.head()

Fetched molecules from ChEMBL: 198


,chembl_id,raw_smiles,max_phase
0,CHEMBL6329,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl,None
1,CHEMBL6328,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1,None
2,CHEMBL265667,Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1,None
3,CHEMBL6362,Cc1ccc(C(=O)c2ccc(-n3ncc(=O)[nH]c3=O)cc2)cc1,None
4,CHEMBL267864,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1,None


In [7]:
from rdkit.Chem import AllChem

curated = []
for _, row in raw_df.iterrows():
    smi = row["raw_smiles"]
    if "." in smi:
        smi = max(smi.split("."), key=len)
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    can_smi = Chem.MolToSmiles(mol, canonical=True)

    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)
    tpsa = rdMolDescriptors.CalcTPSA(mol)

    # Lipinski Rule of 5: <= 1 violation -> drug_like = 1
    violations = sum([mw > 500, logp > 5, hbd > 5, hba > 10])
    drug_like = 1 if violations <= 1 else 0

    # 1024-bit Morgan Fingerprint (radius=2 ~ ECFP4)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)

    curated.append({
        "chembl_id": row["chembl_id"],
        "canonical_smiles": can_smi,
        "mol": mol,
        "mol_wt": mw,
        "mol_logp": logp,
        "hbd": hbd,
        "hba": hba,
        "tpsa": tpsa,
        "drug_like": drug_like,
        "fp": np.array(fp)
    })

curated_df = pd.DataFrame(curated).drop_duplicates(subset=["canonical_smiles"]).reset_index(drop=True)
print("Unique curated molecules:", len(curated_df))
print(curated_df["drug_like"].value_counts())

Unique curated molecules: 198
drug_like
1    178
0     20
Name: count, dtype: int64


In [8]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

# Extract Murcko scaffold SMILES for each molecule
curated_df["scaffold"] = curated_df["mol"].apply(
    lambda m: Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m))
)

X_fp = np.stack(curated_df["fp"].values)
y = curated_df["drug_like"].values
groups = curated_df["scaffold"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_fp, y, groups=groups))

clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
clf.fit(X_fp[train_idx], y[train_idx])

preds = clf.predict(X_fp[test_idx])
probs = clf.predict_proba(X_fp[test_idx])[:, 1]

print("Scaffold-Split Test Accuracy:", round(accuracy_score(y[test_idx], preds), 3))
print("Scaffold-Split ROC-AUC:", round(roc_auc_score(y[test_idx], probs), 3))
print(classification_report(y[test_idx], preds, zero_division=0))

Scaffold-Split Test Accuracy: 0.936
Scaffold-Split ROC-AUC: 0.553
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.94      1.00      0.97        44

    accuracy                           0.94        47
   macro avg       0.47      0.50      0.48        47
weighted avg       0.88      0.94      0.91        47

